# Sin dataset review

The four SURF files from the Minerva publication campaigns, reviewed the same way as the
Neves set: load, look for anything missing or internally inconsistent, attach a Kraken ID to
each ligand, cut down to a clean frame, then plot the distributions.

| file | campaign | selector |
|---|---|---|
| `API_Buchwald_Hartwig/API_BH_ML_HTE_SURF.csv` | API Buchwald-Hartwig | ML |
| `API_Nickel_Suzuki/API_NS_ML_HTE_SURF.csv` | API Nickel-Suzuki | ML |
| `Nickel_Suzuki_1/ML_HTE_SURF.csv` | Nickel-Suzuki 1 | ML |
| `Nickel_Suzuki_1/Experimentalist_HTE_SURF.csv.csv` | Nickel-Suzuki 1 | Experimentalist |

Two things differ from the Neves workflow and drive the design below.

**There is no `ligand_smiles` column.** Neves recovered the Kraken ID by pulling the phosphine
fragment out of `catalyst_smiles`. That does not work here: these campaigns dose the
precatalyst and the ligand separately, so `catalyst_smiles` is the bare Ni/Pd source
(`[Ni(COD)(DQ)]`, `[Ni(oTol)(Cl)(TMEDA)]`, ...) with no phosphine in it at all. What the files
do carry is `ligand_cas`, and Kraken ships three CAS columns -- so CAS is the primary key
here, with a normalised name match as the fallback.

**The four files do not share a schema.** Column counts are 31/31/37/38, the yield column is
`product_1_area` in one file and `product_1_area%` in the others, and only the Nickel_Suzuki_1
pair carries SMILES for the starting materials and product. The loader reconciles these
rather than assuming they line up.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SURF = (Path.home() / "Dropbox" / "Family Room" / "Experimental data" / "Kohen Goble"
        / "ML" / "Extended Ligand Files" / "Sin" / "minerva-main" / "minerva-main"
        / "experimental_campaigns" / "experiments" / "publication" / "SURF_files")

KRAKEN = Path.cwd() / "gp_hub" / "reference" / "kraken" / "kraken_identifiers.csv"
if not KRAKEN.exists():
    KRAKEN = Path.cwd().parent / "gp_hub" / "reference" / "kraken" / "kraken_identifiers.csv"

CAMPAIGN = {
    "API_BH_ML_HTE_SURF.csv":           ("API Buchwald-Hartwig", "ML"),
    "API_NS_ML_HTE_SURF.csv":           ("API Nickel-Suzuki", "ML"),
    "ML_HTE_SURF.csv":                  ("Nickel-Suzuki 1", "ML"),
    "Experimentalist_HTE_SURF.csv.csv": ("Nickel-Suzuki 1", "Experimentalist"),
}

frames = []
FILE_COLUMNS = {}          # the columns each file actually has, captured once
for path in sorted(SURF.rglob("*.csv")):
    raw = pd.read_csv(path)
    rel = path.relative_to(SURF).as_posix()
    FILE_COLUMNS[rel] = list(raw.columns)
    campaign, selector = CAMPAIGN[path.name]
    raw.insert(0, "source_file", path.relative_to(SURF).as_posix())
    raw.insert(1, "campaign", campaign)
    raw.insert(2, "selector", selector)
    frames.append(raw)
    print(f"{path.relative_to(SURF).as_posix():50s} {raw.shape[0]:4d} x {raw.shape[1]}")

df = pd.concat(frames, ignore_index=True)
print(f"\ncombined: {df.shape[0]} rows x {df.shape[1]} columns")

## 1. Review

Read-only. Nothing is dropped until section 3, so these checks describe the data as
published.

### 1a. Schema: which columns exist in which file

A column present in only some files becomes NaN for the others once concatenated, so these
are the columns whose missingness is structural rather than a real gap.

In [ ]:
present = pd.DataFrame({
    f: pd.Series(True, index=cols) for f, cols in FILE_COLUMNS.items()
}).notna().fillna(False)
present.index.name = "column"
partial = present[~present.all(axis=1)]
print("columns NOT shared by all four files:")
print(partial.to_string() if len(partial) else "  (none)")
print(f"\nshared by all four: {len(present) - len(partial)} columns")

### 1b. Missing values

Split into structurally missing (the column does not exist in that file) and genuinely blank
cells, because only the second kind is a data problem.

In [ ]:
rows = []
for f, g in df.groupby("source_file"):
    cols = set(FILE_COLUMNS[f])
    for c in df.columns:
        if c in ("source_file", "campaign", "selector") or c not in cols:
            continue
        n = g[c].isna().sum()
        if n:
            rows.append({"source_file": f, "column": c, "missing": int(n), "of": len(g)})
gaps = pd.DataFrame(rows)
print("genuinely blank cells in columns that DO exist in that file:")
print(gaps.to_string(index=False) if len(gaps) else "  (none)")

### 1c. Name / CAS / SMILES consistency

Three ways the identifiers can disagree: one name carrying several CAS numbers (or the
reverse), one name carrying several SMILES, or a set of solvent fractions that does not sum
to 1.

In [ ]:
def inconsistent(frame, key, value):
    """Keys mapping to more than one distinct non-null value."""
    sub = frame[[key, value]].dropna().drop_duplicates()
    counts = sub.groupby(key)[value].nunique()
    bad = counts[counts > 1]
    return sub[sub[key].isin(bad.index)].sort_values(key)


for key, value in [("ligand_name", "ligand_cas"),
                   ("ligand_cas", "ligand_name"),
                   ("catalyst_name", "catalyst_smiles"),
                   ("reagent_1_name", "reagent_1_smiles"),
                   ("solvent_1_name", "solvent_1_smiles"),
                   ("solvent_2_name", "solvent_2_smiles"),
                   ("startingmat_1_name", "startingmat_1_smiles"),
                   ("product_1_name", "product_1_smiles")]:
    if key not in df.columns or value not in df.columns:
        continue
    bad = inconsistent(df, key, value)
    if len(bad):
        print(f"MISMATCH  {key} -> {value}: {bad[key].nunique()} key(s) with >1 value")
        print(bad.to_string(index=False), "\n")
    else:
        print(f"ok        {key} -> {value}")

In [ ]:
frac_cols = [c for c in df.columns if c.endswith("_fraction")]
total = df[frac_cols].fillna(0).sum(axis=1)
off = df.loc[~np.isclose(total, 1.0), ["source_file", *frac_cols]].assign(total=total)
print(f"solvent fractions not summing to 1: {len(off)} of {len(df)} rows")
if len(off):
    print(off.drop_duplicates().head(12).to_string(index=False))

### 1d. The response column -- two problems

**The name is inconsistent.** `product_1_area` in the BH file, `product_1_area%` in the other
three. Same quantity, two names; unified to `product_area_pct` below.

**The `%` in the name is wrong.** Every one of these columns is a *fraction* on 0-1, not a
percentage on 0-100. The cell below prints the per-file range so that is visible rather than
assumed. It matters: pooling these with a genuinely percentage-scaled dataset, or reading a
mean of 0.28 as "28% yield" without checking, would be an easy and silent mistake.

In [ ]:
YIELD_CANDIDATES = ["product_1_area%", "product_1_area"]
SEL_CANDIDATES = ["selectivity_area%", "selectivity_area"]

print(f"{'file':50s} {'column':22s} {'min':>6s} {'max':>6s} {'mean':>6s}")
for f, g in df.groupby("source_file"):
    for c in [x for x in df.columns if "area" in x.lower()]:
        if c not in FILE_COLUMNS[f]:
            continue
        y = pd.to_numeric(g[c], errors="coerce")
        print(f"{f:50s} {c:22s} {y.min():6.2f} {y.max():6.2f} {y.mean():6.2f}")

df["product_area_pct"] = df[[c for c in YIELD_CANDIDATES if c in df.columns]].bfill(axis=1).iloc[:, 0]
sel = [c for c in SEL_CANDIDATES if c in df.columns]
df["selectivity_area_pct"] = df[sel].bfill(axis=1).iloc[:, 0] if sel else np.nan

y = pd.to_numeric(df.product_area_pct, errors="coerce")
print(f"\nunified product_area_pct: {y.notna().sum()} of {len(df)} rows non-null")
print(f"  range {y.min():.3f} to {y.max():.3f}, mean {y.mean():.3f}")
print(f"  values >1 (would imply a true percentage): {(y > 1).sum()}")
print(f"  negative: {(y < 0).sum()}")

In [ ]:
cols = [c for c in df.columns if c not in ("source_file", "campaign", "selector")]
dups = df[df.duplicated(cols, keep=False)]
print(f"duplicate rows (ignoring the three added columns): {len(dups)} rows "
      f"in {dups.duplicated(cols).sum()} duplicated group(s)")
if len(dups):
    print(dups[["source_file", "ligand_name", "product_area_pct"]].head(10).to_string(index=False))

## 2. Kraken ID

CAS first, because it is unambiguous and covers most of the set. Names are the fallback:
normalised (lowercased, non-alphanumerics stripped), with Kraken's own aliases split on `/`
and `=` so `"(pDMA-Ph)P(tBu)2 = AmPhos"` can reach `AmPhos`.

Nothing is guessed. A ligand matching neither way is left `<NA>` and listed, so the unmatched
set stays visible instead of being silently dropped.

In [ ]:
kraken = pd.read_csv(KRAKEN)

def norm(s):
    return "".join(ch for ch in str(s).lower() if ch.isalnum())

cas_to_id, name_to_id = {}, {}
for col in [c for c in kraken.columns if "cas" in c.lower()]:
    for value, kid in zip(kraken[col], kraken["id"]):
        if pd.notna(value):
            cas_to_id.setdefault(str(value).strip(), int(kid))
for value, kid in zip(kraken["ligand"], kraken["id"]):
    for alias in str(value).replace("=", "/").split("/"):
        if alias.strip():
            name_to_id.setdefault(norm(alias), int(kid))
kraken_name = dict(zip(kraken["id"].astype(int), kraken["ligand"]))

print(f"kraken: {len(kraken)} ligands | {len(cas_to_id)} CAS keys | {len(name_to_id)} name keys")


def kraken_id(name, cas):
    """CAS first, then a normalised name. pd.NA when neither resolves."""
    key = str(cas).strip()
    if key in cas_to_id:
        return cas_to_id[key], "cas"
    hit = name_to_id.get(norm(name))
    if hit is not None:
        return hit, "name"
    return pd.NA, "unmatched"


pairs = df[["ligand_name", "ligand_cas"]].drop_duplicates()
resolved = {(n, c): kraken_id(n, c) for n, c in pairs.itertuples(index=False)}

df["Kraken ID"] = pd.array([resolved[(n, c)][0] for n, c in
                            zip(df.ligand_name, df.ligand_cas)], dtype="Int64")
df["kraken_match"] = [resolved[(n, c)][1] for n, c in zip(df.ligand_name, df.ligand_cas)]
df["Kraken Name"] = df["Kraken ID"].map(kraken_name)

print("\nhow each ROW was matched:")
print(df.kraken_match.value_counts().to_string())
print("\nhow each distinct LIGAND was matched:")
print(pd.Series([v[1] for v in resolved.values()]).value_counts().to_string())

In [ ]:
unmatched = sorted({(n, str(c)) for (n, c), v in resolved.items() if v[1] == "unmatched"})
print(f"{len(unmatched)} ligand(s) with no Kraken entry "
      f"({(df.kraken_match == 'unmatched').sum()} rows):\n")
for n, c in unmatched:
    print(f"  {n:28s} cas={c:16s} {(df.ligand_name == n).sum():4d} rows")

print("\nspot-check, 12 matched ligands:")
print(df[df.kraken_match != "unmatched"][["ligand_name", "ligand_cas", "Kraken ID",
                                          "Kraken Name", "kraken_match"]]
      .drop_duplicates().sort_values("ligand_name").head(12).to_string(index=False))

## 3. Cleaned, down-selected frame

Kept: the reagents, the conditions (both solvents and their fractions), the equivalences, the
product, the response, and the Kraken ID.

Dropped: CAS and SMILES once they have done their job of identification, plus `plate_num` /
`rxn_id` / `vial`, which are plate bookkeeping rather than reaction variables.

`scale_mol`, `concentration_mol_l`, `temperature_deg_c`, `time_h` and `stirring_shaking` are
kept even where constant inside one campaign -- they vary *between* campaigns, and this frame
pools all four files.

In [ ]:
KEEP = [
    "source_file", "campaign", "selector",
    "temperature_deg_c", "time_h", "scale_mol", "stirring_shaking", "concentration_mol_l",
    "startingmat_1_name", "startingmat_1_eq", "startingmat_2_name", "startingmat_2_eq",
    "reagent_1_name", "reagent_1_eq", "reagent_2_name", "reagent_2_eq",
    "catalyst_name", "catalyst_eq", "ligand_name", "ligand_eq",
    "Kraken ID", "Kraken Name", "kraken_match",
    "solvent_1_name", "solvent_1_fraction", "solvent_2_name", "solvent_2_fraction",
    "product_1_name", "product_area_pct", "selectivity_area_pct",
]

df_cleaned = df[[c for c in KEEP if c in df.columns]].copy()

# numeric columns arrive as object once four files are concatenated
for c in ("temperature_deg_c", "time_h", "scale_mol", "concentration_mol_l",
          "startingmat_1_eq", "startingmat_2_eq", "reagent_1_eq", "reagent_2_eq",
          "catalyst_eq", "ligand_eq", "solvent_1_fraction", "solvent_2_fraction",
          "product_area_pct", "selectivity_area_pct"):
    if c in df_cleaned.columns:
        df_cleaned[c] = pd.to_numeric(df_cleaned[c], errors="coerce")

# a single-solvent reaction has no second fraction -- that is 0, not unknown
if "solvent_2_fraction" in df_cleaned.columns:
    df_cleaned.loc[df_cleaned.solvent_2_name.isna(), "solvent_2_fraction"] = 0.0

print(f"{df_cleaned.shape[0]} rows x {df_cleaned.shape[1]} columns")
print("dropped:", sorted(set(df.columns) - set(df_cleaned.columns)))
display(df_cleaned.head())

In [ ]:
print("rows surviving each successive filter:\n")
step = df_cleaned
print(f"  all rows                  {len(step):5d}")
step = step[step.product_area_pct.notna()]
print(f"  with a response           {len(step):5d}")
step = step[step["Kraken ID"].notna()]
print(f"  + ligand found in Kraken  {len(step):5d}")

df_model = step.reset_index(drop=True)
print(f"\nmodelling frame: {df_model.shape[0]} rows x {df_model.shape[1]} columns")
print(f"  campaigns : {sorted(df_model.campaign.unique())}")
print(f"  ligands   : {df_model['Kraken ID'].nunique()} distinct Kraken IDs")
print(f"  response  : {df_model.product_area_pct.min():.3f} to "
      f"{df_model.product_area_pct.max():.3f} (mean {df_model.product_area_pct.mean():.3f})")

### What varies, and where

A column constant across the pooled frame cannot inform a model; one that varies only between
campaigns is a campaign label in disguise. Both are worth seeing before choosing features.

In [ ]:
rows = []
for c in df_model.columns:
    if c in ("source_file", "campaign", "selector"):
        continue
    overall = df_model[c].nunique(dropna=False)
    within = df_model.groupby("campaign")[c].nunique(dropna=False).max()
    rows.append({"column": c, "distinct_overall": overall, "max_within_a_campaign": within,
                 "verdict": ("constant" if overall <= 1 else
                             "campaign-level only" if within <= 1 else "varies")})
variation = pd.DataFrame(rows).sort_values(["verdict", "column"])
print(variation.to_string(index=False))

## 4. Distribution plots

One stacked bar per feature, segments sized by how many reactions use each level. Labels fit
themselves to the segment: name over count where there is room, name beside count rotated
when that renders larger, name alone, then the count alone. The pooled "other" segment never
truncates -- its `(#)` is the whole point of it.

Because the campaigns use different substrates and reagents, each campaign is plotted
separately. A combined view would stack chemistry that never co-occurred.

In [ ]:
import textwrap

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, AutoMinorLocator

FONT_SIZE = 15
LUMPED_FONT_SIZE = 18
LUMPED_MIN_SIZE = 9
NAME_MIN_SIZE = 7
COUNT_MIN_SIZE = 5
LUMPED_PREFIX = "Cat. ("          # shorthand for "categories"

PLOT_COLUMNS = ["Kraken Name", "catalyst_name", "reagent_1_name", "reagent_2_name",
                "startingmat_1_name", "startingmat_2_name", "product_1_name",
                "solvent_1_name", "solvent_2_name", "solvent_1_fraction",
                "temperature_deg_c", "ligand_eq", "catalyst_eq", "concentration_mol_l"]


def top_categories(series, max_n=12, min_n=3, min_frac=0.015):
    counts = series.value_counts(dropna=False)
    frac = counts / counts.sum()
    n_keep = min(max(min_n, int((frac >= min_frac).sum())), max_n, len(counts))
    if len(counts) - n_keep == 1:
        n_keep += 1
    kept, rest = counts.iloc[:n_keep], counts.iloc[n_keep:]
    if len(rest) > 0:
        kept = pd.concat([kept, pd.Series({f"{LUMPED_PREFIX}{len(rest)})": rest.sum()})])
    return kept


def _clip(name, n):
    return name if len(name) <= n else name[:n - 1] + "\u2026"


def _wrap(name, width):
    lines = textwrap.wrap(name, width) or [name]
    if len(lines) > 3:
        lines = lines[:3]
        lines[-1] = lines[-1][:-1] + "\u2026"
    return lines


def label_modes(name, count, lumped=False):
    """Label modes, most complete first; each a list of (text, rotation).

    Inside a mode the variants compete only on font size, so the orientation
    that renders largest wins. Rotated name+count is inline: a second line would
    sit beside the name once rotated, reading as the number being below it.
    """
    count_s = f"{count:,}"
    if lumped:
        # never truncate -- the (#) is the point of a lumped label
        yield [(f"{name}\n{count_s}", 0), (f"{name}  {count_s}", 90)]
        yield [(name, 0), (name, 90)]
        yield [(count_s, 0), (count_s, 90)]
        return
    both = [("\n".join(_wrap(name, w) + [count_s]), 0)
            for w in (len(name), 30, 24, 18, 14, 10)]
    both += [(f"{_clip(name, n)}  {count_s}", 90)
             for n in (len(name), 26, 20, 15, 11, 7)]
    yield both
    yield ([(_clip(name, n), 0) for n in (len(name), 22, 16, 11)]
           + [(_clip(name, n), 90) for n in (len(name), 26, 20, 15, 11, 7, 4)])
    yield [(count_s, 0), (count_s, 90)]


def fit_label(text, rect, name, count, renderer, lumped=False):
    box = rect.get_window_extent(renderer)
    if lumped:
        sizes = range(LUMPED_FONT_SIZE, LUMPED_MIN_SIZE - 1, -1)
    else:
        sizes = range(FONT_SIZE, NAME_MIN_SIZE - 1, -1)
    for variants in label_modes(name, count, lumped):
        for size in sizes:
            text.set_fontsize(size)
            for candidate, rotation in variants:
                text.set_text(candidate)
                text.set_rotation(rotation)
                ext = text.get_window_extent(renderer)
                if ext.width <= box.width - 4 and ext.height <= box.height - 4:
                    return
    text.set_text("")


def distribution_figure(frame, title, width=20):
    """Stacked bar per feature, one row each, labels fitted to their segment."""
    features = [c for c in PLOT_COLUMNS
                if c in frame.columns and frame[c].notna().any() and frame[c].nunique() > 1]
    if not features:
        print(f"{title}: nothing varies")
        return None
    total = len(frame)
    colors = plt.get_cmap("tab20").colors

    fig, ax = plt.subplots(figsize=(width, 2.1 * len(features) + 1))
    segments = []
    for row, feature in enumerate(features):
        left = 0
        for i, (value, count) in enumerate(top_categories(frame[feature]).items()):
            lumped = str(value).startswith(LUMPED_PREFIX)
            name = str(value) if not pd.isna(value) else "NA"
            color = "lightgrey" if lumped else colors[i % len(colors)]
            rect = ax.barh(row, count, left=left, height=0.85, color=color,
                           edgecolor="white", linewidth=1)[0]
            text = ax.text(left + count / 2, row, "", ha="center", va="center",
                           fontsize=FONT_SIZE, rotation_mode="anchor")
            segments.append((rect, text, name, count, lumped))
            left += count

    ax.set_yticks(range(len(features)), labels=features, fontsize=15)
    ax.set_ylim(len(features) - 0.5, -0.5)
    ax.set_xlim(0, total)
    ax.set_xlabel("Number of reactions", fontsize=15)
    ax.set_title(f"{title}  (n = {total:,})", fontsize=17, pad=12)
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(axis="x", which="both", direction="in", labelsize=12)
    ax.tick_params(axis="y", length=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    fig.tight_layout()
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for rect, text, name, count, lumped in segments:
        fit_label(text, rect, name, count, renderer, lumped=lumped)
    return fig

In [ ]:
plt.close("all")
for (campaign, selector), group in df_model.groupby(["campaign", "selector"]):
    fig = distribution_figure(group, f"{campaign} -- {selector}-selected")
    if fig is not None:
        display(fig)
        plt.close(fig)

### Response distributions

The same split, on the response itself. Note the x axis runs 0-1: these are fractions, not
percentages, despite the `%` in the original column names.

In [ ]:
plt.close("all")
groups = list(df_model.groupby(["campaign", "selector"]))
fig, axes = plt.subplots(1, len(groups), figsize=(5.2 * len(groups), 4.0),
                         constrained_layout=True, sharey=True)
for ax, ((campaign, selector), group) in zip(np.atleast_1d(axes), groups):
    y = group.product_area_pct.dropna()
    ax.hist(y, bins=np.linspace(0, 1, 26), color="#4878a8", edgecolor="white")
    ax.axvline(y.mean(), color="#c0392b", lw=2,
               label=f"mean {y.mean():.3f}\nmedian {y.median():.3f}\nn = {len(y):,}")
    ax.set(title=f"{campaign}\n{selector}-selected", xlabel="product area (fraction)",
           xlim=(0, 1))
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("reactions")
display(fig)
plt.close(fig)

print("response by campaign:")
print(df_model.groupby(["campaign", "selector"]).product_area_pct
      .agg(["count", "min", "mean", "median", "max"]).round(3).to_string())

### Response by ligand

Which ligands actually gave product. Ordered by median, so the spread within a ligand is
visible alongside its typical performance.

In [ ]:
plt.close("all")
for (campaign, selector), group in df_model.groupby(["campaign", "selector"]):
    order = (group.groupby("Kraken Name").product_area_pct
             .median().sort_values(ascending=False).index.tolist())
    data = [group.loc[group["Kraken Name"] == lig, "product_area_pct"].dropna().values
            for lig in order]
    fig, ax = plt.subplots(figsize=(max(9, 0.34 * len(order)), 4.6), constrained_layout=True)
    ax.boxplot(data, tick_labels=order, showfliers=False)
    for i, values in enumerate(data, start=1):
        ax.scatter(np.random.default_rng(0).normal(i, 0.055, len(values)), values,
                   s=9, alpha=0.5, color="#c0392b", zorder=3)
    ax.set(title=f"{campaign} -- {selector}-selected   ({len(order)} ligands)",
           ylabel="product area (fraction)", ylim=(-0.03, 1.03))
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    display(fig)
    plt.close(fig)